In [ ]:
# Tạo môi trường
# !python -m venv chatbot_env


In [2]:
# Cài các thư viện cần thiết
!pip install -r ./requirement.txt

  Using cached langchain_core-1.3.1-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.2.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached unstructured-0.22.22-py3-none-any.whl.metadata (29 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.7.35-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Usin


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Xoá môi trường
# !rmdir /s /q chatbot_env

đọc file data

In [3]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader

loader = DirectoryLoader(
  path='./papers',
  glob='**/*.pdf',
  loader_cls=UnstructuredFileLoader,
  show_progress=True,
  use_multithreading=True
)

docs = loader.load()
print(docs)
print(len(docs))

100%|██████████| 2/2 [00:01<00:00,  1.83it/s]

[Document(metadata={'source': 'papers\\article_ai_future.pdf'}, page_content='The Age of Artificial Intelligence: How AI Is Reshaping Our World and What Comes Next\n\nBy Claude | Technology & Society | April 2026\n\nAbstract\n\nArtificial Intelligence has transitioned from a niche academic pursuit into one of the most\n\ntransformative forces of the 21st century. This article examines the current state of AI, its\n\nwide-ranging impact across industries, the ethical dilemmas it presents, and what humanity\n\nmight expect as\n\nthese\n\ntechnologies continue\n\nto accelerate. Drawing on recent\n\ndevelopments in large language models, autonomous systems, and AI-driven healthcare,\n\nwe explore both the promise and the profound responsibility that comes with building\n\nintelligent machines.\n\n1. From Science Fiction to Everyday Reality\n\nNot long ago, the idea of a machine that could write poetry, diagnose cancer, drive a car,\n\nor hold a meaningful conversation existed only in the p

Chunking

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pprint import pprint

MARKDOWN_SEPERATORS = [
  "\n#{1,6}",
  "'''\n",
  "\n\\*\\*\\*+\n",
  "\n---+\n",
  "\n\n",
  "\n",
  " ",
  ""
]

text_spitter = RecursiveCharacterTextSplitter(
  chunk_size = 1200,
  chunk_overlap = 200,
  add_start_index = True,
  strip_whitespace = True,
  separators=MARKDOWN_SEPERATORS
)

splits = text_spitter.split_documents(docs)

pprint(splits)

[Document(metadata={'source': 'papers\\article_ai_future.pdf', 'start_index': 0}, page_content='The Age of Artificial Intelligence: How AI Is Reshaping Our World and What Comes Next\n\nBy Claude | Technology & Society | April 2026\n\nAbstract\n\nArtificial Intelligence has transitioned from a niche academic pursuit into one of the most\n\ntransformative forces of the 21st century. This article examines the current state of AI, its\n\nwide-ranging impact across industries, the ethical dilemmas it presents, and what humanity\n\nmight expect as\n\nthese\n\ntechnologies continue\n\nto accelerate. Drawing on recent\n\ndevelopments in large language models, autonomous systems, and AI-driven healthcare,\n\nwe explore both the promise and the profound responsibility that comes with building\n\nintelligent machines.\n\n1. From Science Fiction to Everyday Reality\n\nNot long ago, the idea of a machine that could write poetry, diagnose cancer, drive a car,\n\nor hold a meaningful conversation exi

embedding model

In [ ]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS #vector database free của fb
from langchain_community.vectorstores.utils import DistanceStrategy

load_dotenv()

embeddings = OpenAIEmbeddings(
  model='text-embedding-3-large'
)

vectorstore = FAISS.from_documents(
  documents=splits, #muốn lưu trữ cái gì
  embedding=embeddings, #embedding bằng cái gì
  distance_strategy=DistanceStrategy.COSINE #thực hiện so sánh 2 vector bằng cái gì 
)

retriever = vectorstore.as_retriever(
  search_type='similarity_score_threshold', #chỉ lấy các chunk có score vượt quá ngưỡng quy định
  search_kwargs={"k":5, "score_threshold": 0.2} #lấy tối đa 5 chunk
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = (
    "You are a strict, citation-focused assistant for a private knowledge base.\n"
    "RULES:\n"
    "1) Use ONLY the provided context to answer.\n"
    "2) If the answer is not clearly contained in the context, say: "
    "\"I don't know based on the provided documents.\"\n"
    "3) Do NOT use outside knowledge, guessing, or web information.\n"
    "4) If applicable, cite sources as (source:page) using the metadata.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
  model='gpt-5-mini',
  temperature=0 #mức độ ngẫu nhiên trong câu trả lời của mô hình
)

Kết nối các phần lại với nhau

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
  {"context": retriever, "question": RunnablePassthrough}
  | prompt
  | llm
  |StrOutputParser()
)

In [ ]:
question = input("Question: ")
answer = rag_chain.invoke(question)
print(answer)